# 12. M1 Hard Normal 시계열 육안 검토

이 노트북은 11번 hard normal 오탐 분석 결과를 기준으로 normal 라벨을 변경하지 않고, hard normal 8건의 7일 원시 시계열과 compact13 feature 패턴을 사람이 검토할 수 있게 정리한다.

고정 기준:
- dataset: `strict_no_event20`
- feature: `compact13`
- threshold: `0.6`
- normal 라벨은 변경하지 않음
- 원본 operational data는 M1만 사용


In [1]:
from pathlib import Path
import zipfile
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=UserWarning)
plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 160

DATA_DIR = Path("05_데이터셋") / "PreDist"
ZIP_PATH = DATA_DIR / "predist_dataset.zip"
NOTEBOOK_PATH = Path("06_노트북") / "12_m1_hard_normal_timeseries_review.ipynb"
OUTPUT_DIR = Path("07_데이터산출물")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

HARD_AUDIT_PATH = OUTPUT_DIR / "m1_hard_normal_audit.csv"
FEATURES_PATH = OUTPUT_DIR / "m1_feature_expansion_features.csv"
FEATURE_SET_SUMMARY_PATH = OUTPUT_DIR / "m1_compact_feature_set_summary.csv"
LABEL_WINDOW_AUDIT_PATH = OUTPUT_DIR / "m1_label_window_audit.csv"

REVIEW_CSV = OUTPUT_DIR / "m1_hard_normal_timeseries_review.csv"
DELTAS_CSV = OUTPUT_DIR / "m1_hard_normal_event_feature_deltas.csv"
DECISION_CSV = OUTPUT_DIR / "m1_hard_normal_review_decision.csv"
REPORT_PATH = OUTPUT_DIR / "12_M1_hard_normal_timeseries_review_보고서.md"

TARGET_EVENTS = [35, 48, 8, 19, 27, 33, 39, 68]
SIGNALS_TO_PLOT = [
    "outdoor_temperature",
    "s_hc1_supply_temperature",
    "s_hc1_supply_temperature_setpoint",
    "s_hc1_supply_temperature_error",
    "p_hc1_return_temperature",
    "p_net_return_temperature",
    "p_return_gap",
    "p_net_meter_flow",
]

SOURCE_PREFIX = "manufacturer 1"


In [2]:
hard_audit = pd.read_csv(HARD_AUDIT_PATH)
features = pd.read_csv(FEATURES_PATH)
feature_set_summary = pd.read_csv(FEATURE_SET_SUMMARY_PATH)
label_window_audit = pd.read_csv(LABEL_WINDOW_AUDIT_PATH)

for frame in [hard_audit, features, label_window_audit]:
    frame["source_event_id"] = frame["source_event_id"].astype(int)
    frame["substation_id"] = frame["substation_id"].astype(int)
    if "window_start" in frame.columns:
        frame["window_start"] = pd.to_datetime(frame["window_start"])
    if "window_end" in frame.columns:
        frame["window_end"] = pd.to_datetime(frame["window_end"])

hard_flag = hard_audit["hard_normal_flag"].astype(str).str.lower().eq("true")
hard_events = hard_audit.loc[hard_flag].copy()
hard_events["source_event_id"] = hard_events["source_event_id"].astype(int)
hard_events = hard_events.set_index("source_event_id").loc[TARGET_EVENTS].reset_index()

compact_row = feature_set_summary.loc[
    feature_set_summary["feature_set"].eq("compact13_overlap")
].iloc[0]
COMPACT13_FEATURES = compact_row["features"].split("|")

assert len(hard_events) == 8, f"hard normal rows mismatch: {len(hard_events)}"
assert set(hard_events["source_event_id"]) == set(TARGET_EVENTS)
assert len(COMPACT13_FEATURES) == 13
assert all(feature in features.columns for feature in COMPACT13_FEATURES)
assert (features.loc[features["source_event_id"].isin([20, 34, 69])].empty)
assert (hard_events["label"] == "normal").all()

hard_events[[
    "source_event_id",
    "substation_id",
    "window_start",
    "window_end",
    "coverage_rate",
    "disturbance_count",
    "hard_normal_strategy",
    "max_compact_probability",
]]


,source_event_id,substation_id,window_start,window_end,coverage_rate,disturbance_count,hard_normal_strategy,max_compact_probability
0,35,11,2017-10-31 00:00:00,2017-11-07 00:00:00,1.0,0,fold_selected_compact13,0.997498
1,48,11,2016-09-30 00:00:00,2016-10-07 00:00:00,1.0,0,fold_selected_compact13,0.775316
2,8,6,2020-06-02 00:00:00,2020-06-09 00:00:00,1.0,0,fold_selected_compact13,0.955289
3,19,8,2018-02-01 00:00:00,2018-02-08 00:00:00,1.0,0,fold_selected_compact13,0.876715
4,27,12,2015-06-24 13:34:00,2015-07-01 13:34:00,1.0,0,locked_compact13,0.603622
5,33,3,2019-01-11 00:00:00,2019-01-18 00:00:00,1.0,0,fold_selected_compact13,0.635146
6,39,15,2018-09-30 00:00:00,2018-10-07 00:00:00,1.0,0,fold_selected_compact13,0.875113
7,68,13,2019-12-20 00:00:00,2019-12-27 00:00:00,1.0,0,locked_compact13,0.688434


## 원본 operational 시계열 로드

원본 ZIP 내부의 M1 operational CSV를 substation 단위로 읽는다. 파일은 세미콜론 구분자이며, 파생 신호 두 개를 원시 시계열 위에서 계산한다.


In [3]:
_operational_cache = {}


def load_operational(substation_id: int) -> pd.DataFrame:
    substation_id = int(substation_id)
    if substation_id in _operational_cache:
        return _operational_cache[substation_id].copy()

    csv_path = f"{SOURCE_PREFIX}/operational_data/substation_{substation_id}.csv"
    with zipfile.ZipFile(ZIP_PATH) as zf:
        with zf.open(csv_path) as fh:
            df = pd.read_csv(fh, sep=";")

    df["timestamp"] = pd.to_datetime(df["timestamp"])
    for col in df.columns:
        if col != "timestamp":
            df[col] = pd.to_numeric(df[col], errors="coerce")

    df["s_hc1_supply_temperature_error"] = (
        df["s_hc1_supply_temperature"] - df["s_hc1_supply_temperature_setpoint"]
    )
    df["p_return_gap"] = (
        df["p_hc1_return_temperature"] - df["p_net_return_temperature"]
    )
    _operational_cache[substation_id] = df
    return df.copy()


def window_slice(substation_id: int, window_start, window_end) -> pd.DataFrame:
    df = load_operational(substation_id)
    mask = (df["timestamp"] >= pd.Timestamp(window_start)) & (
        df["timestamp"] < pd.Timestamp(window_end)
    )
    return df.loc[mask].copy()


sample_window = window_slice(11, hard_events.loc[0, "window_start"], hard_events.loc[0, "window_end"])
sample_window[SIGNALS_TO_PLOT].describe().T[["count", "mean", "std", "min", "max"]]


,count,mean,std,min,max
outdoor_temperature,1008.0,16.326935,3.029612,10.000000,24.000000
s_hc1_supply_temperature,1008.0,46.118527,9.622545,27.400000,74.600000
s_hc1_supply_temperature_setpoint,1008.0,31.256746,19.060684,10.000000,58.000000
s_hc1_supply_temperature_error,1008.0,14.861781,16.938381,-21.800000,44.800000
p_hc1_return_temperature,1008.0,31.371081,4.571129,21.000000,49.500000
p_net_return_temperature,998.0,55.965681,4.680180,38.000000,60.000000
p_return_gap,998.0,-24.558918,7.234314,-34.833333,2.966667
p_net_meter_flow,998.0,99.695641,59.743573,53.000000,566.000000


## compact13 feature 기준 비교

hard normal의 compact13 feature 값이 true negative normal 평균과 positive 평균 중 어느 쪽에 더 가까운지 계산한다. 이 비교는 라벨을 바꾸기 위한 것이 아니라, 왜 threshold 0.6에서 위험처럼 보였는지 설명하기 위한 audit이다.


In [4]:
hard_ids = set(TARGET_EVENTS)
normal_features = features.loc[features["label"].eq("normal")].copy()
hard_feature_rows = normal_features.loc[normal_features["source_event_id"].isin(hard_ids)].copy()
true_negative_features = normal_features.loc[~normal_features["source_event_id"].isin(hard_ids)].copy()
positive_features = features.loc[features["label"].eq("efd_possible")].copy()

delta_rows = []
for _, event_row in hard_feature_rows.sort_values("source_event_id").iterrows():
    event_id = int(event_row["source_event_id"])
    for feature in COMPACT13_FEATURES:
        event_value = float(event_row[feature])
        tn_values = pd.to_numeric(true_negative_features[feature], errors="coerce").dropna()
        positive_values = pd.to_numeric(positive_features[feature], errors="coerce").dropna()
        hard_values = pd.to_numeric(hard_feature_rows[feature], errors="coerce").dropna()
        tn_mean = float(tn_values.mean())
        tn_std = float(tn_values.std(ddof=0))
        positive_mean = float(positive_values.mean())
        hard_mean = float(hard_values.mean())
        if not np.isfinite(tn_std) or tn_std == 0:
            z_vs_tn = np.nan
        else:
            z_vs_tn = (event_value - tn_mean) / tn_std
        diff_tn = abs(event_value - tn_mean)
        diff_positive = abs(event_value - positive_mean)
        closer_group = "true_negative" if diff_tn <= diff_positive else "positive"
        delta_rows.append(
            {
                "source_event_id": event_id,
                "substation_id": int(event_row["substation_id"]),
                "feature": feature,
                "event_value": event_value,
                "true_negative_mean": tn_mean,
                "true_negative_std": tn_std,
                "hard_normal_mean": hard_mean,
                "positive_mean": positive_mean,
                "z_vs_true_negative": z_vs_tn,
                "abs_z_vs_true_negative": abs(z_vs_tn) if np.isfinite(z_vs_tn) else np.nan,
                "abs_diff_to_true_negative": diff_tn,
                "abs_diff_to_positive": diff_positive,
                "closer_group": closer_group,
            }
        )

deltas = pd.DataFrame(delta_rows)
deltas["rank_abs_z_within_event"] = deltas.groupby("source_event_id")[
    "abs_z_vs_true_negative"
].rank(method="first", ascending=False)
deltas["is_top_deviation"] = deltas["rank_abs_z_within_event"].le(3)
deltas.to_csv(DELTAS_CSV, index=False, encoding="utf-8-sig")

deltas.head()


,source_event_id,substation_id,feature,event_value,true_negative_mean,true_negative_std,hard_normal_mean,positive_mean,z_vs_true_negative,abs_z_vs_true_negative,abs_diff_to_true_negative,abs_diff_to_positive,closer_group,rank_abs_z_within_event,is_top_deviation
0,8,6,outdoor_temperature__last_12h_mean_minus_prev_...,-0.172894,1.603507,2.132112,0.581253,-1.008904,-0.833164,0.833164,1.776400,0.836011,positive,3.0,True
1,8,6,outdoor_temperature__last_6h_mean_minus_prev_6...,-3.078009,-1.945243,1.766834,-1.147286,1.215749,-0.641127,0.641127,1.132766,4.293758,true_negative,6.0,False
2,8,6,outdoor_temperature__last_minus_first,-0.070000,-2.506019,3.352302,0.062500,0.188095,0.726670,0.726670,2.436019,0.258095,positive,4.0,False
3,8,6,p_hc1_return_temperature__last_12h_mean_minus_...,0.666875,0.318824,2.434180,0.732786,-1.641260,0.142985,0.142985,0.348051,2.308135,true_negative,10.0,False
4,8,6,p_hc1_return_temperature__last_1d_mean_minus_p...,1.935238,0.736708,1.757429,0.199405,-3.560078,0.681980,0.681980,1.198531,5.495316,true_negative,5.0,False


## 이벤트별 시계열 PNG 생성

각 hard normal 이벤트의 7일 window를 같은 축 구성으로 저장한다. 그림 제목에는 threshold 0.6에서 오탐으로 잡힌 근거 probability와 compact13 상위 deviation feature를 같이 표시한다.


In [5]:
def short_feature_name(feature: str) -> str:
    return (
        feature.replace("s_hc1_supply_temperature_setpoint", "supply_setpoint")
        .replace("s_hc1_supply_temperature_error", "supply_error")
        .replace("s_hc1_supply_temperature", "supply_temp")
        .replace("p_hc1_return_temperature", "hc1_return")
        .replace("p_net_return_temperature", "net_return")
        .replace("p_net_meter_flow", "flow")
        .replace("outdoor_temperature", "outdoor")
        .replace("p_return_gap", "return_gap")
        .replace("__", "\n")
        .replace("_minus_", "-")
        .replace("_", " ")
    )


def top_deviation_text(event_id: int, top_n: int = 3) -> str:
    subset = deltas.loc[deltas["source_event_id"].eq(event_id)]
    subset = subset.sort_values("abs_z_vs_true_negative", ascending=False).head(top_n)
    return "; ".join(
        f"{short_feature_name(row.feature)} z={row.abs_z_vs_true_negative:.2f}"
        for row in subset.itertuples()
    )


def plot_event_timeseries(event_row: pd.Series) -> Path:
    event_id = int(event_row["source_event_id"])
    substation_id = int(event_row["substation_id"])
    window = window_slice(substation_id, event_row["window_start"], event_row["window_end"])
    png_path = OUTPUT_DIR / f"m1_hard_normal_event_{event_id:04d}_timeseries.png"

    fig, axes = plt.subplots(5, 1, figsize=(14, 11), sharex=True)
    fig.suptitle(
        f"Hard normal event {event_id} | substation {substation_id} | "
        f"p={event_row['max_compact_probability']:.3f} | {top_deviation_text(event_id)}",
        fontsize=10,
    )

    axes[0].plot(window["timestamp"], window["outdoor_temperature"], color="#666666", lw=1.1)
    axes[0].set_ylabel("outdoor")
    axes[0].grid(alpha=0.25)

    axes[1].plot(window["timestamp"], window["s_hc1_supply_temperature"], label="supply", color="#1f77b4", lw=1.1)
    axes[1].plot(window["timestamp"], window["s_hc1_supply_temperature_setpoint"], label="setpoint", color="#ff7f0e", lw=1.0)
    axes[1].set_ylabel("supply")
    axes[1].legend(loc="upper right", fontsize=8)
    axes[1].grid(alpha=0.25)

    axes[2].plot(window["timestamp"], window["s_hc1_supply_temperature_error"], color="#d62728", lw=1.0)
    axes[2].axhline(0, color="#333333", lw=0.8, alpha=0.6)
    axes[2].set_ylabel("supply error")
    axes[2].grid(alpha=0.25)

    axes[3].plot(window["timestamp"], window["p_hc1_return_temperature"], label="hc1 return", color="#2ca02c", lw=1.0)
    axes[3].plot(window["timestamp"], window["p_net_return_temperature"], label="net return", color="#9467bd", lw=1.0)
    axes[3].set_ylabel("return temp")
    axes[3].legend(loc="upper right", fontsize=8)
    axes[3].grid(alpha=0.25)

    axes[4].plot(window["timestamp"], window["p_return_gap"], color="#8c564b", lw=1.0, label="return gap")
    axes[4].set_ylabel("return gap")
    axes[4].grid(alpha=0.25)
    ax2 = axes[4].twinx()
    ax2.plot(window["timestamp"], window["p_net_meter_flow"], color="#17becf", lw=0.9, alpha=0.75, label="flow")
    ax2.set_ylabel("flow")

    for ax in axes:
        ax.margins(x=0)
    fig.autofmt_xdate()
    fig.tight_layout(rect=[0, 0.02, 1, 0.95])
    fig.savefig(png_path, bbox_inches="tight")
    plt.close(fig)
    return png_path


event_png_paths = {}
for _, row in hard_events.iterrows():
    event_png_paths[int(row["source_event_id"])] = plot_event_timeseries(row)

event_png_paths


{35: WindowsPath('07_데이터산출물/m1_hard_normal_event_0035_timeseries.png'),
 48: WindowsPath('07_데이터산출물/m1_hard_normal_event_0048_timeseries.png'),
 8: WindowsPath('07_데이터산출물/m1_hard_normal_event_0008_timeseries.png'),
 19: WindowsPath('07_데이터산출물/m1_hard_normal_event_0019_timeseries.png'),
 27: WindowsPath('07_데이터산출물/m1_hard_normal_event_0027_timeseries.png'),
 33: WindowsPath('07_데이터산출물/m1_hard_normal_event_0033_timeseries.png'),
 39: WindowsPath('07_데이터산출물/m1_hard_normal_event_0039_timeseries.png'),
 68: WindowsPath('07_데이터산출물/m1_hard_normal_event_0068_timeseries.png')}

In [6]:
def plot_substation11_comparison() -> Path:
    rows = hard_events.loc[hard_events["source_event_id"].isin([35, 48])].copy()
    rows = rows.sort_values("source_event_id")
    png_path = OUTPUT_DIR / "m1_hard_normal_substation11_event35_48_comparison.png"
    fig, axes = plt.subplots(5, 2, figsize=(15, 11), sharex=False)

    for col_idx, (_, event_row) in enumerate(rows.iterrows()):
        event_id = int(event_row["source_event_id"])
        window = window_slice(event_row["substation_id"], event_row["window_start"], event_row["window_end"])
        axes[0, col_idx].set_title(
            f"Event {event_id} | p={event_row['max_compact_probability']:.3f}",
            fontsize=10,
        )
        axes[0, col_idx].plot(window["timestamp"], window["outdoor_temperature"], color="#666666", lw=1.0)
        axes[0, col_idx].set_ylabel("outdoor")

        axes[1, col_idx].plot(window["timestamp"], window["s_hc1_supply_temperature"], label="supply", color="#1f77b4", lw=1.0)
        axes[1, col_idx].plot(window["timestamp"], window["s_hc1_supply_temperature_setpoint"], label="setpoint", color="#ff7f0e", lw=1.0)
        axes[1, col_idx].set_ylabel("supply")
        axes[1, col_idx].legend(fontsize=8)

        axes[2, col_idx].plot(window["timestamp"], window["s_hc1_supply_temperature_error"], color="#d62728", lw=1.0)
        axes[2, col_idx].axhline(0, color="#333333", lw=0.8, alpha=0.6)
        axes[2, col_idx].set_ylabel("supply error")

        axes[3, col_idx].plot(window["timestamp"], window["p_hc1_return_temperature"], label="hc1 return", color="#2ca02c", lw=1.0)
        axes[3, col_idx].plot(window["timestamp"], window["p_net_return_temperature"], label="net return", color="#9467bd", lw=1.0)
        axes[3, col_idx].set_ylabel("return temp")
        axes[3, col_idx].legend(fontsize=8)

        axes[4, col_idx].plot(window["timestamp"], window["p_return_gap"], color="#8c564b", lw=1.0)
        axes[4, col_idx].set_ylabel("return gap")
        ax2 = axes[4, col_idx].twinx()
        ax2.plot(window["timestamp"], window["p_net_meter_flow"], color="#17becf", lw=0.9, alpha=0.75)
        ax2.set_ylabel("flow")

        for ax in axes[:, col_idx]:
            ax.grid(alpha=0.25)
            ax.margins(x=0)

    fig.suptitle("Substation 11 hard normal comparison: Event 35 vs 48", fontsize=12)
    fig.autofmt_xdate()
    fig.tight_layout(rect=[0, 0.02, 1, 0.96])
    fig.savefig(png_path, bbox_inches="tight")
    plt.close(fig)
    return png_path


comparison_png_path = plot_substation11_comparison()
comparison_png_path


WindowsPath('07_데이터산출물/m1_hard_normal_substation11_event35_48_comparison.png')

In [7]:
def plot_compact13_group_comparison() -> Path:
    png_path = OUTPUT_DIR / "m1_hard_normal_compact13_group_comparison.png"
    tn_mean = true_negative_features[COMPACT13_FEATURES].mean()
    tn_std = true_negative_features[COMPACT13_FEATURES].std(ddof=0).replace(0, np.nan)
    group_means = pd.DataFrame(
        {
            "true_negative": true_negative_features[COMPACT13_FEATURES].mean(),
            "hard_normal": hard_feature_rows[COMPACT13_FEATURES].mean(),
            "positive": positive_features[COMPACT13_FEATURES].mean(),
        }
    )
    z_group = ((group_means.T - tn_mean) / tn_std).replace([np.inf, -np.inf], np.nan)
    plot_values = z_group.fillna(0).to_numpy()

    fig, ax = plt.subplots(figsize=(13, 4.8))
    im = ax.imshow(plot_values, aspect="auto", cmap="coolwarm", vmin=-3, vmax=3)
    ax.set_yticks(range(len(z_group.index)))
    ax.set_yticklabels(z_group.index)
    ax.set_xticks(range(len(COMPACT13_FEATURES)))
    ax.set_xticklabels([short_feature_name(f) for f in COMPACT13_FEATURES], rotation=70, ha="right", fontsize=7)
    ax.set_title("compact13 group means as z-score vs true negative normal")
    fig.colorbar(im, ax=ax, fraction=0.025, pad=0.02, label="z vs true negative")
    fig.tight_layout()
    fig.savefig(png_path, bbox_inches="tight")
    plt.close(fig)
    return png_path


group_comparison_png_path = plot_compact13_group_comparison()
group_comparison_png_path


WindowsPath('07_데이터산출물/m1_hard_normal_compact13_group_comparison.png')

## 이벤트별 판단 생성

판단은 normal 라벨을 바꾸지 않는 audit 태그다. coverage 또는 disturbance 문제가 있으면 `review_required_normal`, 반복 substation 또는 compact13 편차가 있으면 `hard_normal_metadata`, 나머지는 `keep_normal`으로 둔다.


In [8]:
def classify_review_label(
    disturbance_count: int,
    coverage_rate: float,
    same_substation_hard_count: int,
    max_abs_z: float,
    positive_like_count: int,
    sustained_shift_count: int,
) -> tuple[str, str]:
    if disturbance_count > 0:
        return "review_required_normal", "disturbance_count가 0이 아니므로 normal 유지 전 수동 검토 필요"
    if coverage_rate < 0.95:
        return "review_required_normal", "window coverage가 0.95 미만이므로 시계열 품질 검토 필요"
    if same_substation_hard_count >= 2:
        return "hard_normal_metadata", "같은 substation에서 hard normal이 반복되어 설비 고유 정상 패턴 후보"
    if positive_like_count >= 5:
        return "review_required_normal", "compact13 feature 다수가 positive 평균에 더 가까워 후속 수동 검토 필요"
    if max_abs_z >= 2.0 or sustained_shift_count >= 3 or positive_like_count >= 2:
        return "hard_normal_metadata", "compact13 변화량이 true negative normal 평균에서 벗어나 모델 해석용 태그 유지"
    return "keep_normal", "true negative normal 패턴에 더 가까워 normal 유지"


hard_substation_counts = hard_events.groupby("substation_id")["source_event_id"].nunique().to_dict()
normal_substation_events = (
    normal_features.groupby("substation_id")["source_event_id"]
    .apply(lambda values: "|".join(str(int(v)) for v in sorted(values)))
    .to_dict()
)
label_window_normal = label_window_audit.loc[label_window_audit["label"].eq("normal")].copy()

review_rows = []
for _, event_row in hard_events.iterrows():
    event_id = int(event_row["source_event_id"])
    substation_id = int(event_row["substation_id"])
    window = window_slice(substation_id, event_row["window_start"], event_row["window_end"])
    event_deltas = deltas.loc[deltas["source_event_id"].eq(event_id)].copy()
    top_features = (
        event_deltas.sort_values("abs_z_vs_true_negative", ascending=False)
        .head(3)["feature"]
        .tolist()
    )
    positive_like_count = int(event_deltas["closer_group"].eq("positive").sum())
    true_negative_like_count = int(event_deltas["closer_group"].eq("true_negative").sum())
    max_abs_z = float(event_deltas["abs_z_vs_true_negative"].max())
    sustained_keywords = ("last_1d", "last_12h", "last_6h", "last_minus_first")
    sustained_shift_count = int(
        event_deltas.loc[
            event_deltas["feature"].str.contains("|".join(sustained_keywords), regex=True)
            & event_deltas["abs_z_vs_true_negative"].ge(1.5)
        ].shape[0]
    )
    disturbance_count = int(event_row["disturbance_count"])
    label_window_match = label_window_normal.loc[
        label_window_normal["source_event_id"].eq(event_id), "disturbance_count"
    ]
    if not label_window_match.empty:
        disturbance_count = max(disturbance_count, int(label_window_match.iloc[0]))

    same_substation_hard_count = int(hard_substation_counts.get(substation_id, 0))
    review_label, review_reason = classify_review_label(
        disturbance_count=disturbance_count,
        coverage_rate=float(event_row["coverage_rate"]),
        same_substation_hard_count=same_substation_hard_count,
        max_abs_z=max_abs_z,
        positive_like_count=positive_like_count,
        sustained_shift_count=sustained_shift_count,
    )
    closer_summary = (
        "positive_like"
        if positive_like_count > true_negative_like_count
        else "true_negative_like"
    )
    spike_like_flag = bool(max_abs_z < 2.0 and sustained_shift_count <= 2)
    sustained_shift_flag = bool(sustained_shift_count >= 3)

    review_rows.append(
        {
            "source_event_id": event_id,
            "substation_id": substation_id,
            "window_start": event_row["window_start"],
            "window_end": event_row["window_end"],
            "label": event_row["label"],
            "review_label": review_label,
            "review_reason": review_reason,
            "hard_normal_severity": event_row["hard_normal_severity"],
            "hard_normal_strategy": event_row["hard_normal_strategy"],
            "max_compact_probability": float(event_row["max_compact_probability"]),
            "coverage_rate": float(event_row["coverage_rate"]),
            "disturbance_count": disturbance_count,
            "raw_row_count": int(len(window)),
            "expected_sample_count": int(event_row["expected_sample_count"]),
            "top_deviation_features": "|".join(top_features),
            "top_deviation_features_short": "|".join(short_feature_name(f).replace("\n", " ") for f in top_features),
            "positive_like_feature_count": positive_like_count,
            "true_negative_like_feature_count": true_negative_like_count,
            "compact13_closer_summary": closer_summary,
            "max_abs_z_vs_true_negative": max_abs_z,
            "sustained_shift_feature_count": sustained_shift_count,
            "spike_like_flag": spike_like_flag,
            "sustained_shift_flag": sustained_shift_flag,
            "same_substation_hard_normal_count": same_substation_hard_count,
            "same_substation_normal_event_ids": normal_substation_events.get(substation_id, ""),
            "event_png_path": str(event_png_paths[event_id].as_posix()),
        }
    )

review_df = pd.DataFrame(review_rows).sort_values(
    ["review_label", "same_substation_hard_normal_count", "max_compact_probability"],
    ascending=[False, False, False],
)
review_df.to_csv(REVIEW_CSV, index=False, encoding="utf-8-sig")

review_df[[
    "source_event_id",
    "substation_id",
    "review_label",
    "max_compact_probability",
    "coverage_rate",
    "disturbance_count",
    "positive_like_feature_count",
    "max_abs_z_vs_true_negative",
    "top_deviation_features_short",
]]


,source_event_id,substation_id,review_label,max_compact_probability,coverage_rate,disturbance_count,positive_like_feature_count,max_abs_z_vs_true_negative,top_deviation_features_short
3,19,8,review_required_normal,0.876715,1.0,0,5,1.717035,outdoor last-first|outdoor last 12h mean-prev ...
7,68,13,review_required_normal,0.688434,1.0,0,7,2.211561,hc1 return last 1d mean-prev 6d mean|flow last...
0,35,11,hard_normal_metadata,0.997498,1.0,0,4,3.587100,supply setpoint last 1d mean-prev 6d mean|supp...
1,48,11,hard_normal_metadata,0.775316,1.0,0,4,2.041295,net return last 1d mean-prev 6d mean|hc1 retur...
2,8,6,hard_normal_metadata,0.955289,1.0,0,4,1.793742,supply error last-first|supply setpoint last 1...
6,39,15,hard_normal_metadata,0.875113,1.0,0,2,3.466841,supply setpoint last 1d mean-prev 6d mean|supp...
5,33,3,hard_normal_metadata,0.635146,1.0,0,2,2.164555,outdoor last 6h mean-prev 6h mean|flow last 1d...
4,27,12,hard_normal_metadata,0.603622,1.0,0,3,2.624417,outdoor last 6h mean-prev 6h mean|outdoor last...


In [9]:
review_required_count = int(review_df["review_label"].eq("review_required_normal").sum())
hard_metadata_count = int(review_df["review_label"].eq("hard_normal_metadata").sum())
keep_normal_count = int(review_df["review_label"].eq("keep_normal").sum())
substation11_hard_count = int(
    review_df.loc[review_df["substation_id"].eq(11), "source_event_id"].nunique()
)

if review_required_count > 0:
    final_decision = "일부 event는 review_required_normal로 후속 수동 검토"
elif substation11_hard_count >= 2:
    final_decision = "substation 11 정상 패턴 보정 필요"
else:
    final_decision = "normal 유지 + hard_normal metadata만 유지"

decision_rows = [
    {
        "decision_item": "final_decision",
        "decision_value": final_decision,
        "note": "normal 라벨은 변경하지 않고 hard normal audit 태그만 추가",
    },
    {
        "decision_item": "normal_label_policy",
        "decision_value": "keep_company_normal_label",
        "note": "회사 제공 normal label 유지",
    },
    {
        "decision_item": "hard_normal_event_count",
        "decision_value": str(len(review_df)),
        "note": "Event 35, 48, 8, 19, 27, 33, 39, 68",
    },
    {
        "decision_item": "review_required_count",
        "decision_value": str(review_required_count),
        "note": "coverage 또는 disturbance 또는 positive-like 과다 기준",
    },
    {
        "decision_item": "hard_normal_metadata_count",
        "decision_value": str(hard_metadata_count),
        "note": "정상 라벨 유지, 모델 해석용 hard normal tag 유지",
    },
    {
        "decision_item": "keep_normal_count",
        "decision_value": str(keep_normal_count),
        "note": "추가 tag 없이 normal 유지",
    },
    {
        "decision_item": "substation11_event35_48",
        "decision_value": "reviewed",
        "note": "같은 substation에서 반복 FP가 발생해 별도 비교 PNG 생성",
    },
]
decision_df = pd.DataFrame(decision_rows)
decision_df.to_csv(DECISION_CSV, index=False, encoding="utf-8-sig")
decision_df


,decision_item,decision_value,note
0,final_decision,일부 event는 review_required_normal로 후속 수동 검토,normal 라벨은 변경하지 않고 hard normal audit 태그만 추가
1,normal_label_policy,keep_company_normal_label,회사 제공 normal label 유지
2,hard_normal_event_count,8,"Event 35, 48, 8, 19, 27, 33, 39, 68"
3,review_required_count,2,coverage 또는 disturbance 또는 positive-like 과다 기준
4,hard_normal_metadata_count,6,"정상 라벨 유지, 모델 해석용 hard normal tag 유지"
5,keep_normal_count,0,추가 tag 없이 normal 유지
6,substation11_event35_48,reviewed,같은 substation에서 반복 FP가 발생해 별도 비교 PNG 생성


## 보고서 생성


In [10]:
def markdown_table(df: pd.DataFrame, columns: list[str]) -> str:
    header = "| " + " | ".join(columns) + " |"
    sep = "| " + " | ".join(["---"] * len(columns)) + " |"
    rows = []
    for _, row in df[columns].iterrows():
        rows.append("| " + " | ".join(str(row[col]) for col in columns) + " |")
    return "\n".join([header, sep] + rows)


event_table = review_df.copy()
event_table["window_start"] = pd.to_datetime(event_table["window_start"]).dt.strftime("%Y-%m-%d %H:%M")
event_table["window_end"] = pd.to_datetime(event_table["window_end"]).dt.strftime("%Y-%m-%d %H:%M")
event_table["max_compact_probability"] = event_table["max_compact_probability"].map(lambda v: f"{v:.3f}")
event_table["coverage_rate"] = event_table["coverage_rate"].map(lambda v: f"{v:.3f}")
event_table["max_abs_z_vs_true_negative"] = event_table["max_abs_z_vs_true_negative"].map(lambda v: f"{v:.2f}")

event_table_md = markdown_table(
    event_table.sort_values("source_event_id"),
    [
        "source_event_id",
        "substation_id",
        "window_start",
        "window_end",
        "review_label",
        "coverage_rate",
        "disturbance_count",
        "max_compact_probability",
        "positive_like_feature_count",
        "max_abs_z_vs_true_negative",
        "review_reason",
    ],
)

top_reason_rows = (
    review_df[["source_event_id", "top_deviation_features_short", "compact13_closer_summary"]]
    .sort_values("source_event_id")
    .copy()
)
top_reason_md = markdown_table(
    top_reason_rows,
    ["source_event_id", "top_deviation_features_short", "compact13_closer_summary"],
)

png_rows = []
for event_id in TARGET_EVENTS:
    png_rows.append(
        {
            "항목": f"Event {event_id}",
            "파일": Path(event_png_paths[event_id]).name,
        }
    )
png_rows.extend(
    [
        {"항목": "Event 35/48 비교", "파일": comparison_png_path.name},
        {"항목": "compact13 그룹 비교", "파일": group_comparison_png_path.name},
    ]
)
png_md = markdown_table(pd.DataFrame(png_rows), ["항목", "파일"])

report = f"""# M1 Hard Normal 시계열 육안 검토 보고서

## 개요

이번 단계는 학습이 아니라 `compact13 + threshold 0.6` 기준에서 hard normal로 잡힌 normal 8건의 7일 원시 시계열을 사람이 검토할 수 있게 정리한 것이다. 회사 제공 normal 라벨은 변경하지 않았다.

최종 결론: **{final_decision}**

## 무엇을 했는지

- Event 35, 48을 우선 검토하고, 나머지 hard normal Event 8, 19, 27, 33, 39, 68도 같은 방식으로 확인했다.
- 원본 operational CSV에서 7일 window를 다시 읽어 이벤트별 시계열 PNG를 만들었다.
- threshold 0.6에서 FP 근거가 된 compact13 feature 편차를 true negative normal 및 positive 평균과 비교했다.
- `keep_normal`, `hard_normal_metadata`, `review_required_normal` 중 하나로 audit tag를 부여했다.

## 이벤트별 판단

{event_table_md}

## 주요 판단 근거

{top_reason_md}

## Substation 11 Event 35/48

Event 35와 48은 모두 `substation_id=11`에서 발생한 hard normal이다. 두 이벤트 모두 disturbance count가 0이고 coverage가 충분하므로 normal 라벨을 유지한다. 다만 같은 substation에서 반복적으로 threshold 0.6 이상 위험 확률이 나온 만큼, 이 substation의 정상 운전 패턴이 compact13 feature에서 positive와 일부 비슷하게 보일 수 있다. 따라서 현재 결론은 삭제나 재라벨링이 아니라 `hard_normal_metadata`로 관리하는 것이다.

## 생성 이미지

{png_md}

## 변경 내용

| 항목 | 내용 |
| --- | --- |
| 노트북 | `06_노트북/12_m1_hard_normal_timeseries_review.ipynb` |
| 검토 CSV | `07_데이터산출물/m1_hard_normal_timeseries_review.csv` |
| feature delta CSV | `07_데이터산출물/m1_hard_normal_event_feature_deltas.csv` |
| decision CSV | `07_데이터산출물/m1_hard_normal_review_decision.csv` |
| 보고서 | `07_데이터산출물/12_M1_hard_normal_timeseries_review_보고서.md` |

## 검증

```mermaid
flowchart TD
  A[11번 hard normal audit] --> B[원본 M1 operational 7일 window 재로딩]
  B --> C[이벤트별 시계열 PNG 생성]
  A --> D[compact13 feature delta 계산]
  C --> E[review decision 작성]
  D --> E
  E --> F[CSV와 보고서 검증]
```

- hard normal 8건이 모두 review CSV에 포함됐다.
- Event 35/48 별도 비교 PNG를 생성했다.
- normal 라벨 값은 변경하지 않았다.
- 8개 이벤트의 disturbance count는 모두 0으로 재확인됐다.
- Event 20, 34, 69는 판단 dataset에 포함되지 않았다.
- 비대상 제조사 문자열이 산출물에 들어가지 않았는지 검사했다.

## 한계와 주의점

- 이번 판단은 모델 재학습이나 threshold 변경이 아니다.
- `hard_normal_metadata`는 정상 라벨을 유지하면서 해석용으로 남기는 tag다.
- 시계열 육안 검토만으로 설비 원인을 확정하지 않는다. 특히 substation 11은 정상 패턴 보정 후보로 다음 실험에서 별도 feature 또는 substation-aware 해석을 검토할 필요가 있다.

## 다음에 볼 것

- substation 11 정상 window를 더 확보할 수 있는지 확인한다.
- hard normal과 positive를 가르는 feature가 최근 6시간/12시간 변화에 과도하게 의존하는지 점검한다.
- normal 라벨은 유지한 채, hard normal metadata를 학습 평가 리포트에 계속 표시한다.
"""

REPORT_PATH.write_text(report, encoding="utf-8")
print(REPORT_PATH)


07_데이터산출물\12_M1_hard_normal_timeseries_review_보고서.md


## 검증


In [11]:
png_paths = list(event_png_paths.values()) + [comparison_png_path, group_comparison_png_path]
missing_png = [path for path in png_paths if not Path(path).exists() or Path(path).stat().st_size == 0]

assert REVIEW_CSV.exists()
assert DELTAS_CSV.exists()
assert DECISION_CSV.exists()
assert REPORT_PATH.exists()
assert len(review_df) == 8
assert set(review_df["source_event_id"]) == set(TARGET_EVENTS)
assert (review_df["label"] == "normal").all()
assert review_df["disturbance_count"].sum() == 0
assert not missing_png, missing_png
assert len(deltas) == 8 * 13
assert features.loc[features["source_event_id"].isin([20, 34, 69])].empty
assert comparison_png_path.exists() and comparison_png_path.stat().st_size > 0
assert group_comparison_png_path.exists() and group_comparison_png_path.stat().st_size > 0

report_text = REPORT_PATH.read_text(encoding="utf-8")
for event_id in TARGET_EVENTS:
    assert f"Event {event_id}" in report_text or f"| {event_id} |" in report_text

forbidden_terms = ["manufacturer" + "_2", "manufacturer " + "2", "M" + "2"]
check_paths = [NOTEBOOK_PATH, REPORT_PATH, REVIEW_CSV, DELTAS_CSV, DECISION_CSV]
hits = []
for path in check_paths:
    text = Path(path).read_text(encoding="utf-8", errors="ignore")
    if any(term in text for term in forbidden_terms):
        hits.append(str(path))
assert not hits, "비대상 제조사 문자열 발견"

validation_summary = {
    "review_rows": len(review_df),
    "delta_rows": len(deltas),
    "png_count": len(png_paths),
    "disturbance_sum": int(review_df["disturbance_count"].sum()),
    "normal_label_unchanged": bool((review_df["label"] == "normal").all()),
    "final_decision": final_decision,
}
validation_summary


{'review_rows': 8,
 'delta_rows': 104,
 'png_count': 10,
 'disturbance_sum': 0,
 'normal_label_unchanged': True,
 'final_decision': '일부 event는 review_required_normal로 후속 수동 검토'}